# Final Facility Tier Model: K-Means Care-Bundle Clustering

This notebook creates the final data/model artifact for the New Jersey Behavioral Health Access Navigator. It follows the project scope in `README.md` and `docs/project_context.md`: use a small processed/demo dataset, engineer service-related features, run K-Means clustering, and translate numeric clusters into plain-language facility tiers.

The model is an interpretation layer for navigators. It does **not** show real-time appointment availability, book care, decide clinical appropriateness, or replace professional judgment. A care coordinator, social worker, discharge planner, or community navigator should treat the tier as a quick summary of service complexity and then verify details with the facility or official sources.

## Plain-language model summary

K-Means is an unsupervised clustering method. That means it looks for groups of facilities with similar service patterns without being told the correct answer in advance. In this notebook, each facility becomes a row of service signals, such as outpatient care, residential care, medication management, crisis support, payment options, age groups, and ancillary services.

The model groups facilities into care-bundle tiers. After clustering, we inspect each group's feature profile and give it a human-readable label. The labels are meant to help users scan options faster, not to rank facilities from best to worst.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

sns.set_theme(style='whitegrid', palette='Set2')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'facility_service_demo.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'facility_tiers_demo.csv'

DATA_PATH

## Load the small processed/demo dataset

The repository intentionally stores only a small processed demo file. Raw GB-scale source files, ZIP archives, and local databases should stay out of GitHub. The demo columns mirror the type of cleaned SAMHSA-centered facility data expected by the final navigator: facility identity, geography, service categories, payment signals, age groups, ancillary supports, and source-confidence notes.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}')
df.head()

## Engineer service features

Most service columns contain multiple values separated by semicolons. We split those lists and one-hot encode each service signal. The model does not use facility names, IDs, or latitude/longitude as clustering features because the tier should summarize **what services a facility appears to offer**, not simply where it is or what it is called.

We also add simple count features, such as how many service settings or payment options appear in the processed record. These counts help distinguish focused outpatient providers from broader care hubs.

In [ ]:
MULTILABEL_COLUMNS = [
    'service_setting',
    'care_types',
    'treatment_approaches',
    'recovery_support',
    'emergency_services',
    'payment_options',
    'age_groups',
    'ancillary_services',
]

def split_values(value):
    if pd.isna(value) or str(value).strip() == '':
        return []
    return [item.strip() for item in str(value).split(';') if item.strip()]

feature_frames = []
count_features = pd.DataFrame(index=df.index)

for column in MULTILABEL_COLUMNS:
    lists = df[column].apply(split_values)
    count_features[f'{column}_count'] = lists.apply(len)

    mlb = MultiLabelBinarizer()
    encoded = pd.DataFrame(
        mlb.fit_transform(lists),
        columns=[f'{column}__{label}' for label in mlb.classes_],
        index=df.index,
    )
    feature_frames.append(encoded)

service_features = pd.concat(feature_frames + [count_features], axis=1)
service_features['total_service_signal_count'] = count_features.sum(axis=1)

scaler = StandardScaler()
X = scaler.fit_transform(service_features)

print(f'Engineered features: {service_features.shape[1]}')
service_features.head()

## Choose a practical number of clusters

The elbow chart shows how much within-cluster variation remains as we add more clusters. The silhouette score shows whether facilities are more similar to their assigned cluster than to other clusters. For this final prototype, three clusters are a practical choice because they are easy to explain to non-coders and map naturally to focused, expanded, and comprehensive service bundles.

In [ ]:
candidate_ks = range(2, min(8, len(df) - 1) + 1)
model_scores = []

for k in candidate_ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    model_scores.append({
        'k': k,
        'inertia': model.inertia_,
        'silhouette': silhouette_score(X, labels),
    })

scores = pd.DataFrame(model_scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=scores, x='k', y='inertia', marker='o', ax=axes[0])
axes[0].set_title('Elbow check')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Within-cluster variation')

sns.lineplot(data=scores, x='k', y='silhouette', marker='o', ax=axes[1])
axes[1].set_title('Silhouette check')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette score')

plt.tight_layout()
scores

## Fit final K-Means model

We use `k=3` for the final care-bundle tiers. This does not mean there are exactly three types of behavioral health facilities in New Jersey. It means three groups are a simple, explainable summary for the prototype. Future work can revisit the number of clusters when the team has the full cleaned SAMHSA/NPPES/HRSA dataset ready.

In [ ]:
N_CLUSTERS = 3

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
df['cluster'] = kmeans.fit_predict(X)

profile_input = pd.concat([df[['facility_id', 'facility_name', 'county', 'city', 'cluster']], service_features], axis=1)
cluster_profile = profile_input.groupby('cluster')[service_features.columns].mean()

complexity_by_cluster = cluster_profile['total_service_signal_count'].sort_values()
tier_names = [
    'Focused Outpatient Access',
    'Expanded Community Treatment',
    'Comprehensive High-Need Care Hub',
]
cluster_to_tier = {cluster: tier for cluster, tier in zip(complexity_by_cluster.index, tier_names)}

df['facility_tier'] = df['cluster'].map(cluster_to_tier)
df[['facility_id', 'facility_name', 'county', 'city', 'facility_tier', 'cluster']].sort_values(['facility_tier', 'facility_name'])

## Explain what defines each tier

The table below compares each cluster with the overall average feature profile. A positive difference means that feature is more common in that tier than in the demo dataset overall. This is how the team can defend the plain-language labels: the labels come from the service patterns we can observe, not from a clinical quality judgment.

In [ ]:
overall_profile = service_features.mean()
explanation_rows = []

for cluster_id, row in cluster_profile.iterrows():
    lift = (row - overall_profile).sort_values(ascending=False)
    defining_features = [
        feature.replace('__', ': ').replace('_', ' ')
        for feature in lift.head(8).index
    ]
    tier = cluster_to_tier[cluster_id]
    explanation_rows.append({
        'facility_tier': tier,
        'cluster': cluster_id,
        'facilities_in_demo': int((df['cluster'] == cluster_id).sum()),
        'average_service_signal_count': round(row['total_service_signal_count'], 1),
        'plain_language_interpretation': {
            'Focused Outpatient Access': 'Facilities with narrower outpatient-oriented service bundles. Useful starting points for routine counseling, therapy, medication management, or referral conversations when crisis or residential services are not the main need.',
            'Expanded Community Treatment': 'Facilities with broader outpatient or day-treatment bundles, often combining mental health, substance use, recovery, care coordination, and payment-access signals.',
            'Comprehensive High-Need Care Hub': 'Facilities with the broadest service bundles in the demo data, often including crisis, residential, hospital-based, or intensive settings plus multiple support services.',
        }[tier],
        'top_defining_features': '; '.join(defining_features),
    })

tier_explanations = pd.DataFrame(explanation_rows).sort_values('average_service_signal_count')
tier_explanations

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='facility_tier', y=service_features['total_service_signal_count'])
plt.title('Service signal count by assigned facility tier')
plt.xlabel('Facility tier')
plt.ylabel('Total service signal count')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()

## Facility tier output

The final output keeps the original facility information and adds two model columns: `cluster`, the numeric K-Means group, and `facility_tier`, the plain-language label. The app or report should show the plain-language tier, not just the numeric cluster. The numeric cluster has no human meaning by itself.

In [ ]:
output_columns = [
    'facility_id', 'facility_name', 'county', 'city', 'latitude', 'longitude',
    'facility_tier', 'cluster', 'service_setting', 'care_types',
    'emergency_services', 'payment_options', 'age_groups',
    'ancillary_services', 'source_confidence',
]

tiered_facilities = df[output_columns].sort_values(['facility_tier', 'county', 'facility_name'])
tiered_facilities.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote tiered demo output to: {OUTPUT_PATH}')
tiered_facilities.head(10)

## Interpretation and responsible use

Use the tier as a compact summary of service complexity. A `Comprehensive High-Need Care Hub` is not automatically better than a `Focused Outpatient Access` provider. It may simply offer a broader bundle of services. For a person who needs routine outpatient therapy close to home, a focused outpatient facility may be more appropriate than a broader hub.

This model should be paired with human review and source verification. The navigator should still communicate uncertainty around stale records, missing insurance details, source disagreement, geographic access, transportation barriers, and whether a facility is accepting new clients.

## Next steps for the team

1. Replace the demo CSV with the final small processed SAMHSA-centered dataset when available.
2. Re-run the notebook top-to-bottom and compare the elbow/silhouette plots.
3. Re-inspect cluster profiles before keeping or changing the tier names.
4. Add the `facility_tier` field to the Streamlit prototype as an interpretation label.
5. Document any source-confidence or missing-data caveats beside the tier label.